In [ ]:
!pip install transformers peft trl bitsandbytes accelerate datasets wandb -q
!huggingface-cli login
!wandb login

In [ ]:
import os
import torch
import wandb
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from huggingface_hub import login

login()

In [ ]:
# ── 1. CONFIG ────────────────────────────────────────────────
MODEL_ID    = "meta-llama/Llama-3.2-3B-Instruct"
OUTPUT_DIR  = "./nepali-legal-llama"
RANK        = 16
ALPHA       = 32
LR          = 2e-4
EPOCHS      = 3
MAX_SEQ_LEN = 512
BATCH_SIZE  = 4
GRAD_ACCUM  = 4

os.environ["WANDB_PROJECT"]  = "llm-proj"
os.environ["WANDB_RUN_NAME"] = f"llama3.2-3b-r{RANK}-lr{LR}-epoch{EPOCHS}"

In [ ]:
# ── 2. QUANTIZATION ──────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

In [ ]:
# ── 3. TOKENIZER ─────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
# ── 4. BASE MODEL ────────────────────────────────────────────
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False

In [ ]:
# ── 5. LORA / PEFT WRAPPING ──────────────────────────────────  ← THIS WAS MISSING
lora_config = LoraConfig(
    r=RANK,
    lora_alpha=ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # should show ~1-2% trainable

In [ ]:
# ── 6. LOAD DATASET ──────────────────────────────────────────
train_dataset = load_dataset("json", data_files="train.jsonl", split="train")
val_dataset   = load_dataset("json", data_files="val.jsonl",   split="train")

# Sanity check
print(f"Train: {len(train_dataset)} examples")
print(f"Val:   {len(val_dataset)} examples")
print("Sample:", train_dataset[0])

In [ ]:
print("Text field preview:")
print(train_dataset[0]["text"][:200])

In [ ]:
# ── 8. TRAINING CONFIG ───────────────────────────────────────
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    optim="paged_adamw_32bit",
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="wandb",
    dataset_text_field="text",
    max_length=MAX_SEQ_LEN,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
)

In [ ]:
# ── 9. TRAIN ─────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

trainer.train()

In [ ]:
# ── 10. SAVE ADAPTER ─────────────────────────────────────────
trainer.save_model(f"{OUTPUT_DIR}/final-adapter")

In [ ]:
# ── 11. MERGE AND PUSH ───────────────────────────────────────
merged_model = model.merge_and_unload()

In [ ]:
!zip -r final-adapter.zip {OUTPUT_DIR}/final-adapter
from google.colab import files
files.download("final-adapter.zip")